In [1]:
import requests

In [2]:
## reponse = requests.get("http://localhost:7071/api/hedge_trigger")
reponse = requests.get("http://localhost:7071/api/FTSO_Feed")
print(reponse.json())
print(type(reponse.json()))

ConnectionError: HTTPConnectionPool(host='localhost', port=7071): Max retries exceeded with url: /api/FTSO_Feed (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001B3A31D5490>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [ ]:

reponse = requests.get("http://localhost:7071/api/FTSO_Feed_ETH")
print(reponse.json())
print(type(reponse.json()))

{'symbol': 'ETH/USD', 'price': 2091.532, 'decimals': 3, 'timestamp': 1770491969}
<class 'dict'>


In [ ]:
reponse = requests.get("http://localhost:7071/api/hedge_create_message")
print(reponse.json())

{'message': 'ETH holding above 2090—momentum building. Look for a breakout over 2120 or hedge downside below 2050.', 'mood': 'bullish', 'confidence': 0.78, 'slug_found': True, 'market': {'question': 'Will Ethereum reach $7,000 by December 31, 2026?', 'coin': 'ETH', 'yesPrice': 0.65, 'successful_slug': 'will-ethereum-reach-7000-by-december-31-2026'}}


In [ ]:
reponse = requests.get("https://gamma-api.polymarket.com/events/slug/will-ethereum-reach-5000-in-february-2026")
print(reponse.json())
print(reponse.status_code)
print(type(reponse.json()))
print(reponse.json().get("slug"))
print(reponse.json().get("title"))

{'type': 'not found error', 'error': 'slug not found'}
404
<class 'dict'>
None
None


In [ ]:
reponse = requests.get("https://gamma-api.polymarket.com/markets/slug/will-ethereum-reach-5000-in-february-2026")
print(reponse.json())
print(reponse.status_code)
print(type(reponse.json()))

{'id': '1303353', 'question': 'Will Ethereum reach $5,000 in February?', 'conditionId': '0x113263410d37b2256691b62d76b09a97d4ecd75bdcb0df37ff14a7adf53039bb', 'slug': 'will-ethereum-reach-5000-in-february-2026', 'endDate': '2026-03-01T05:00:00Z', 'liquidity': '377318.05411', 'startDate': '2026-01-31T02:16:07.41398Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/ETH+fullsize.jpg', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/ETH+fullsize.jpg', 'description': 'This market will immediately resolve to "Yes" if any Binance 1 minute candle for ETH/USDT during the month specified in the title (from 00:00 AM ET on the first day to 11:59 PM ET on the last), has a final High price equal to or greater than the price specified in the title. Otherwise, this market will resolve to "No."\n\nThe resolution source for this market is Binance, specifically the ETH/USDT High prices available at https://www.binance.com/en/trade/ETH_USDT, with the chart settings on "1m" for

In [ ]:
from __future__ import annotations

from typing import Any, Dict, List, Optional
import requests


GAMMA_BASE_URL = "https://gamma-api.polymarket.com"

In [ ]:
def _get_tag_id(tag_slug: str, session: Optional[requests.Session] = None) -> int:
    """
    Resolve a Polymarket tag slug (e.g., 'ethereum') to its numeric tag id.
    Endpoint: GET https://gamma-api.polymarket.com/tags/slug/{slug}
    """
    s = session or requests.Session()
    url = f"{GAMMA_BASE_URL}/tags/slug/{tag_slug}"
    resp = s.get(url, timeout=20)
    resp.raise_for_status()
    data = resp.json()
    if "id" not in data:
        raise ValueError(f"Tag slug '{tag_slug}' did not return an 'id'. Response: {data}")
    return int(data["id"])
def list_open_ethereum_market_slugs_and_titles(
    *,
    tag_slug: str = "ethereum",
    page_limit: int = 200,
) -> List[Dict[str, str]]:
    """
    Returns ONLY OPEN Polymarket markets for the Ethereum tag.

    Open == active AND not closed.
    Output format:
        [{"slug": "...", "title": "..."}, ...]
    """
    results: List[Dict[str, str]] = []

    with requests.Session() as session:
        tag_id = _get_tag_id(tag_slug, session)

        offset = 0
        while True:
            params: Dict[str, Any] = {
                "tag_id": tag_id,
                "active": True,     # market is live
                "closed": False,    # not resolved/ended
                "limit": page_limit,
                "offset": offset,
            }

            resp = session.get(
                f"{GAMMA_BASE_URL}/markets",
                params=params,
                timeout=30,
            )
            resp.raise_for_status()
            markets = resp.json()

            if not markets:
                break

            for m in markets:
                slug = m.get("slug")
                title = m.get("question")
                if slug and title:
                    results.append({
                        "slug": slug,
                        "title": title,
                    })

            if len(markets) < page_limit:
                break
            offset += page_limit

    return results

In [ ]:
pairs = list_open_ethereum_market_slugs_and_titles()
print(len(pairs))
print(pairs[:5])

382
[{'slug': 'us-national-ethereum-reserve-before-2027', 'title': 'US national Ethereum reserve before 2027?'}, {'slug': 'will-ethereum-reach-10000-by-december-31-2026', 'title': 'Will Ethereum reach $10,000 by December 31, 2026?'}, {'slug': 'will-ethereum-reach-8000-by-december-31-2026', 'title': 'Will Ethereum reach $8,000 by December 31, 2026?'}, {'slug': 'will-ethereum-reach-7500-by-december-31-2026', 'title': 'Will Ethereum reach $7,500 by December 31, 2026?'}, {'slug': 'will-ethereum-reach-7000-by-december-31-2026', 'title': 'Will Ethereum reach $7,000 by December 31, 2026?'}]
